# Batik Multi-Region — Captioning Notebook (Qwen3-VL)

Notebook ini **hanya untuk discovery dataset + auto-captioning**, terpisah dari notebook training. Alasannya: siklus iterasi beda (captioning cukup sekali, training diulang per daerah), dan supaya tidak buang kuota GPU training untuk hal yang seharusnya sekali jalan.

**Output notebook ini** (folder `/kaggle/working/captioned-dataset/`) akan dipakai sebagai **input** di notebook training terpisah, lewat fitur Kaggle *"Add Input → Notebook Output Files"* (pilih notebook ini setelah kamu Save Version), atau kamu simpan output-nya jadi Kaggle Dataset baru kalau mau lebih permanen.

**Yang perlu kamu siapkan:**
1. Tambahkan dataset gambar batik kamu sebagai input (tombol *Add Input*).
2. Cukup GPU **T4 x1** sudah cukup untuk captioning (tidak perlu x2, captioning tidak di-parallel di notebook ini).
3. Setelah selesai, klik **Save Version** supaya output-nya bisa dipakai sebagai input notebook training.

## 1. Install dependency

Pillow di-pin `<12` karena versi 12.0.0 punya bug import (`_Ink` dari `PIL._typing`) yang lagi ramai dilaporkan di banyak environment Colab/Kaggle per akhir 2025 — kalau tidak dipin, `pip install -U` bisa menarik versi itu dan bikin error saat proses caption. `transformers>=4.57.0` wajib untuk Qwen3-VL (nama arsitektur internalnya beda dari Qwen2.5-VL).

In [1]:
!pip install -U "transformers>=4.57.0" accelerate bitsandbytes "qwen-vl-utils>=0.0.14" "pillow<12" --break-system-packages -q
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

import transformers
print("transformers version:", transformers.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 40.4 MB/s eta 0:00:00
CUDA available: True
GPU: Tesla T4
transformers version: 5.14.1


## 2. Discover region dataset

Scan semua folder di tiap `DATASET_ROOTS`, pakai nama folder sebagai label region, dan kumpulkan semua gambar (rekursif, support banyak sub-folder). Kalau satu region ada di lebih dari satu dataset root, nama file diberi prefix `srcN_` biar tidak bentrok.

Path di bawah pakai format `/kaggle/input/datasets/<pemilik>/<slug-dataset>/...` — ini format yang benar untuk dataset yang ditambahkan lewat Kaggle Datasets (beda dengan competition data yang langsung di `/kaggle/input/<slug>/`).

In [2]:
import os

# ==================== SESUAIKAN INI ====================
DATASET_ROOTS = [
    "/kaggle/input/datasets/yohanpermanautm/citra-batik-madura-and-flora",
    "/kaggle/input/datasets/hydiexe/dataset-fix/dataset_batik_fix/raw_dataset",
    "/kaggle/input/datasets/reyhanfisena/dataset-batik-jawa/dataset_batik",
]
REGION_LABEL_OVERRIDES = {}   # contoh: {"NamaFolderAneh": "solo"}
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
LIMIT_IMAGES_PER_REGION_FOR_TEST = None   # isi angka kecil (mis. 5) dulu untuk tes cepat, None = semua
# ==========================================================

def auto_clean_label(name):
    label = name.strip().lower()
    for junk in ["batik_", "batik-", "batik "]:
        if label.startswith(junk):
            label = label[len(junk):]
            break
    label = label.strip().replace(" ", "_").replace("-", "_")
    while "__" in label:
        label = label.replace("__", "_")
    return label.strip("_")

def discover_regions(dataset_roots):
    regions = {}
    for root in dataset_roots:
        if not os.path.isdir(root):
            print(f"[!] Root tidak ditemukan, dilewati: {root}")
            continue
        for entry in sorted(os.listdir(root)):
            full = os.path.join(root, entry)
            if not os.path.isdir(full):
                continue
            label = REGION_LABEL_OVERRIDES.get(entry, auto_clean_label(entry))
            regions.setdefault(label, []).append(full)
    return regions

def find_images_recursive(root):
    found = []
    for dirpath, _dirnames, filenames in os.walk(root):
        for fn in filenames:
            if os.path.splitext(fn)[1].lower() in IMG_EXTENSIONS:
                found.append(os.path.join(dirpath, fn))
    return sorted(found)

region_sources = discover_regions(DATASET_ROOTS)
print(f"Region terdeteksi: {len(region_sources)}\n")
for label, srcs in sorted(region_sources.items()):
    print(f"  - {label}  <-  {srcs}")

region_file_entries = {}
for label, src_dirs in region_sources.items():
    entries = []
    multi_source = len(src_dirs) > 1
    for i, src_dir in enumerate(src_dirs):
        for img_path in find_images_recursive(src_dir):
            rel = os.path.relpath(img_path, src_dir)
            flat = rel.replace(os.sep, "_")
            if multi_source:
                flat = f"src{i}_{flat}"
            entries.append((flat, img_path))
    if LIMIT_IMAGES_PER_REGION_FOR_TEST:
        entries = entries[:LIMIT_IMAGES_PER_REGION_FOR_TEST]
    region_file_entries[label] = entries
    print(f"{label}: {len(entries)} gambar siap diproses")

ALL_REGIONS = sorted(region_file_entries.keys())
TRIGGER_MAP = {region: f"{region}batik" for region in ALL_REGIONS}
print("\nTrigger token per region:", TRIGGER_MAP)


Region terdeteksi: 28

  - betawi  <-  ['/kaggle/input/datasets/hydiexe/dataset-fix/dataset_batik_fix/raw_dataset/batik_betawi']
  - bokor_kencono  <-  ['/kaggle/input/datasets/hydiexe/dataset-fix/dataset_batik_fix/raw_dataset/batik_bokor_kencono']
  - buketan  <-  ['/kaggle/input/datasets/hydiexe/dataset-fix/dataset_batik_fix/raw_dataset/batik_buketan']
  - dayak  <-  ['/kaggle/input/datasets/hydiexe/dataset-fix/dataset_batik_fix/raw_dataset/batik_dayak']
  - flora  <-  ['/kaggle/input/datasets/yohanpermanautm/citra-batik-madura-and-flora/Flora']
  - jawa_barat  <-  ['/kaggle/input/datasets/reyhanfisena/dataset-batik-jawa/dataset_batik/Jawa_Barat']
  - jawa_tengah  <-  ['/kaggle/input/datasets/reyhanfisena/dataset-batik-jawa/dataset_batik/Jawa_Tengah']
  - jawa_timur  <-  ['/kaggle/input/datasets/reyhanfisena/dataset-batik-jawa/dataset_batik/Jawa_Timur']
  - jlamprang  <-  ['/kaggle/input/datasets/hydiexe/dataset-fix/dataset_batik_fix/raw_dataset/batik_jlamprang']
  - kawung  <-  ['/k

## 3. Load Qwen3-VL-8B-Instruct (4-bit)

Pakai class generic `AutoModelForImageTextToText` (bukan nama class spesifik seperti `Qwen2_5_VLForConditionalGeneration`) supaya kode ini lebih tahan kalau Alibaba merilis versi Qwen-VL yang lebih baru lagi — tinggal ganti `MODEL_ID`.

`min_pixels`/`max_pixels` di bawah pakai kelipatan 32 (bukan 28 seperti di Qwen2.5-VL) karena Qwen3-VL membulatkan resolusi ke kelipatan 32.

In [3]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

caption_processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*32*32, max_pixels=1024*32*32)
caption_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="cuda:0",
)
caption_model.eval()
print("Qwen3-VL siap.")


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Qwen3-VL siap.


## 4. Generate caption

Prompt sengaja **tidak** minta model menyebut nama region/motif spesifik atau kata "batik" — supaya caption murni deskriptif visual, dan identitas region sepenuhnya dipegang oleh *trigger token* yang di-prepend otomatis ke tiap caption.

Rentang kata (`CAPTION_TARGET_LENGTH`) tetap dipakai sebagai batas aman (biar tidak kepotong token limit CLIP di SDXL / tidak jadi cuma 3 kata), tapi prompt sekarang eksplisit bilang: **pakai bagian bawah rentang untuk motif sederhana, bagian atas untuk motif rumit/berlapis** — supaya panjang caption jujur mengikuti kompleksitas asli gambar, bukan model 'mengarang' detail cuma demi mengejar jumlah kata.

In [4]:
CAPTION_TARGET_LENGTH = "15-30 words"   # naikkan mis. "30-60 words" kalau base model training-nya pakai T5 (FLUX/Qwen-Image)

CAPTION_PROMPT = (
    "You are labeling a photo of an Indonesian batik textile for an image-generation training "
    f"dataset. In ONE fluent English sentence, describe ONLY what is visibly on the cloth: the "
    "pattern style (e.g. repeating diagonal lines, circular medallions, floral sprigs, cloud-like "
    "swirls, dots, geometric grid), the dominant colors, and the density/scale of the motif. "
    f"Keep the sentence within {CAPTION_TARGET_LENGTH}: use the SHORTER end if the pattern is simple "
    "or repetitive, and the LONGER end only if the pattern genuinely has multiple distinct layers or "
    "elements worth describing — do not pad with invented detail just to reach a length. Do NOT name "
    "a specific region, city, or traditional motif name, and do NOT use the word 'batik' itself — just "
    "describe the visual pattern as if describing plain fabric."
)

@torch.no_grad()
def generate_caption(image_path):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": CAPTION_PROMPT},
        ],
    }]
    text = caption_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = caption_processor(text=[text], images=image_inputs, videos=video_inputs,
                                padding=True, return_tensors="pt").to(caption_model.device)
    output_ids = caption_model.generate(**inputs, max_new_tokens=100, do_sample=False)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)]
    caption = caption_processor.batch_decode(trimmed, skip_special_tokens=True,
                                               clean_up_tokenization_spaces=True)[0].strip()
    return caption

import shutil

SKIP_IF_CAPTION_EXISTS = True
CAPTIONED_ROOT = "/kaggle/working/captioned-dataset"

for region, entries in region_file_entries.items():
    trigger = TRIGGER_MAP[region]
    out_dir = os.path.join(CAPTIONED_ROOT, region)
    os.makedirs(out_dir, exist_ok=True)
    n_done, n_skipped = 0, 0
    for flat_name, src_path in entries:
        stem, ext = os.path.splitext(flat_name)
        dst_img = os.path.join(out_dir, flat_name)
        dst_txt = os.path.join(out_dir, stem + ".txt")

        if not os.path.exists(dst_img):
            shutil.copy(src_path, dst_img)

        if SKIP_IF_CAPTION_EXISTS and os.path.exists(dst_txt):
            n_skipped += 1
            continue

        raw_caption = generate_caption(src_path)
        full_caption = f"{trigger} batik, {raw_caption}"
        with open(dst_txt, "w", encoding="utf-8") as f:
            f.write(full_caption)
        n_done += 1

    print(f"[{region}] caption baru: {n_done}, dilewati (sudah ada): {n_skipped}")


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:944: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


[madura_pola_kompleks] caption baru: 150, dilewati (sudah ada): 0
[madura_pola_sederhana] caption baru: 138, dilewati (sudah ada): 12
[flora] caption baru: 150, dilewati (sudah ada): 0
[betawi] caption baru: 150, dilewati (sudah ada): 0
[bokor_kencono] caption baru: 150, dilewati (sudah ada): 0
[buketan] caption baru: 150, dilewati (sudah ada): 0
[dayak] caption baru: 124, dilewati (sudah ada): 26
[jlamprang] caption baru: 150, dilewati (sudah ada): 0
[kawung] caption baru: 150, dilewati (sudah ada): 0
[liong] caption baru: 150, dilewati (sudah ada): 0
[mega_mendung] caption baru: 150, dilewati (sudah ada): 0
[parang] caption baru: 150, dilewati (sudah ada): 0
[sekarjagad] caption baru: 150, dilewati (sudah ada): 0
[sidoluhur] caption baru: 150, dilewati (sudah ada): 0
[sidomukti] caption baru: 150, dilewati (sudah ada): 0
[sidomulyo] caption baru: 150, dilewati (sudah ada): 0
[singa_barong] caption baru: 150, dilewati (sudah ada): 0
[srikaton] caption baru: 150, dilewati (sudah ada): 

## 5. Sanity check hasil caption

**Wajib cek manual** beberapa hasil sebelum training penuh — VLM bisa saja salah baca motif yang rumit/tumpang tindih, dan untuk lomba, akurasi deskripsi visual berpengaruh ke kualitas LoRA. Cek juga apakah panjang caption memang bervariasi sesuai kompleksitas (bukan selalu mepet ke batas atas).

In [5]:
import random

for region in ALL_REGIONS:
    out_dir = os.path.join(CAPTIONED_ROOT, region)
    txt_files = [f for f in os.listdir(out_dir) if f.endswith(".txt")]
    if not txt_files:
        continue
    sample = random.choice(txt_files)
    with open(os.path.join(out_dir, sample), encoding="utf-8") as f:
        content = f.read()
    print(f"[{region}] {sample} ({len(content.split())} kata):\n  {content}\n")


[betawi] 7.txt (21 kata):
  betawibatik batik, Vibrant red fabric with dense, repeating blue and yellow motifs of horses, carriages, and stylized floral and starburst patterns.

[bokor_kencono] -59-_jpg.rf.6d5e5a9c0784d22c95c0326409a1a4da.txt (21 kata):
  bokor_kenconobatik batik, Repeating oval medallions with intricate inner swirls, in deep navy, beige, and cream, densely packed across the fabric surface.

[buketan] buketan_222.txt (24 kata):
  buketanbatik batik, Rich floral sprigs in coral, blue, and cream densely cover a dark field, mirrored on a soft peach background with scattered butterflies.

[dayak] images-4-_jpg.rf.0fbc8e066893b2924bacb08ffe41b97f.txt (25 kata):
  dayakbatik batik, Repeating floral sprigs and swirling tendrils in cream, gold, and red on a deep navy background, densely packed in a flowing, interwoven pattern.

[flora] anggrek (21).txt (22 kata):
  florabatik batik, Repeating floral sprigs in soft white and magenta, densely packed with delicate petals and vibr

## 6. Bebaskan VRAM & siapkan output untuk notebook training

Setelah ini, klik **Save Version** di kanan atas. Di notebook training, tambahkan input lewat *Add Input → Notebook Output Files* dan pilih notebook ini — folder `captioned-dataset/` akan muncul di `/kaggle/input/<slug-notebook-ini>/captioned-dataset/`.

In [6]:
del caption_model
torch.cuda.empty_cache()

total_images = sum(len([f for f in os.listdir(os.path.join(CAPTIONED_ROOT, r)) if not f.endswith('.txt')]) for r in ALL_REGIONS)
print(f"Selesai. Total {total_images} gambar+caption di {CAPTIONED_ROOT}, siap dipakai notebook training.")
print("Regions:", ALL_REGIONS)


Selesai. Total 3850 gambar+caption di /kaggle/working/captioned-dataset, siap dipakai notebook training.
Regions: ['betawi', 'bokor_kencono', 'buketan', 'dayak', 'flora', 'jawa_barat', 'jawa_tengah', 'jawa_timur', 'jlamprang', 'kawung', 'liong', 'madura', 'madura_pola_kompleks', 'madura_pola_sederhana', 'mega_mendung', 'parang', 'sekarjagad', 'sidoluhur', 'sidomukti', 'sidomulyo', 'singa_barong', 'srikaton', 'tribusono', 'tujuh_rupa', 'tuntrum', 'wahyu_tumurun', 'wirasat', 'yogyakarta']
